# Zlab Cloud · Straight BallSprint
Use a GPU runtime with Internet access. The first block trains braking and stable stops at a 2.8 m/s command cap.

In [ ]:
import os
from pathlib import Path
import subprocess
import tempfile

repo = Path(tempfile.mkdtemp(prefix="zlab-cloud-")) / "Zlab-cloud"
subprocess.run([
    "git", "clone", "--depth", "1", "--branch", "main",
    "https://github.com/hafizadha/Zlab-cloud.git", str(repo),
], check=True)
project = repo / "Zlab_Gym"
launcher = project / "train_straight_ball_sprint.sh"
if not launcher.is_file():
    raise RuntimeError("This checkout does not contain the active straight BallSprint task.")
subprocess.run(["git", "-C", str(repo), "log", "-1", "--oneline"], check=True)


import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[huggingface]", "onnx"], cwd=project, check=True)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU accelerator before training.")


In [ ]:
# Set this to False to keep checkpoints only in Kaggle.
# For uploads, add a write token in Kaggle Secrets.
UPLOAD_TO_HF = True

if UPLOAD_TO_HF:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["ZLAB_HF_REPO_ID"] = "hafiz24/zlab-checkpoints"


In [ ]:
UPDATES = 500
ENVS_PER_GPU = 256
COMMAND_SPEED = 2.8
TORSO_PUSH_SCALE = 0.0
RESUME_CHECKPOINT = ""  # Absolute path to a selected checkpoint from this task.
HF_RESUME_FILE = ""  # e.g. checkpoints/<run>/model_100.pt

log_root = Path("/kaggle/working/zlab-straight-ball-sprint")
run_env = dict(
    os.environ,
    ZLAB_LOG_ROOT=str(log_root),
    ZLAB_USE_ALL_GPUS="1",
    ZLAB_PYTHON=sys.executable,
    ZLAB_COMMAND_SPEED=str(COMMAND_SPEED),
    ZLAB_TORSO_PUSH_SCALE=str(TORSO_PUSH_SCALE),
)
args = [str(launcher), str(UPDATES), str(ENVS_PER_GPU)]
if RESUME_CHECKPOINT and HF_RESUME_FILE:
    raise ValueError("Choose either a local checkpoint or a Hugging Face checkpoint.")
resume_path = RESUME_CHECKPOINT
if HF_RESUME_FILE:
    from huggingface_hub import hf_hub_download

    resume_path = hf_hub_download(
        repo_id=os.environ["ZLAB_HF_REPO_ID"], filename=HF_RESUME_FILE
    )
if resume_path:
    args.append(str(Path(resume_path).expanduser().resolve(strict=True)))
subprocess.run(args, cwd=project, env=run_env, check=True)


In [ ]:
# Paste the checkpoint path printed by the run, such as model_100.pt.
CHECKPOINT = ""
EVAL_SEED = 2026
EVAL_TORSO_PUSH_SCALE = TORSO_PUSH_SCALE
if CHECKPOINT:
    subprocess.run([
        sys.executable, "scripts/play.py",
        "--checkpoint-file", str(Path(CHECKPOINT).expanduser().resolve(strict=True)),
        "--device", "cuda:0", "--eval-episodes", "48", "--num-envs", "48",
        "--seed", str(EVAL_SEED), "--command-speed", str(COMMAND_SPEED),
        "--disturbances", "True", "--torso-push-scale", str(EVAL_TORSO_PUSH_SCALE),
    ], cwd=project, check=True)
